<a href="https://colab.research.google.com/github/kimimilo/hello-world-/blob/main/DAA_Topic3_Lab1_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# This code sets up an in-memory SQLite database using Python and connects it to pandas. It is useful for auditing dataset bias because you can store, query, and analyze datasets using SQL directly inside a Jupyter Notebook

import sqlite3
import pandas as pd

# An in-memory SQLite database: it lives only while this notebook runtime is running
conn = sqlite3.connect(":memory:")

def sql(query):
    """Run a SELECT query and show the result as a table."""
    return pd.read_sql_query(query, conn)

def run(statement):
    """Run statements that change the database (CREATE, INSERT, UPDATE, DELETE)."""
    conn.executescript(statement)
    conn.commit()

print("SQLite version:", sqlite3.sqlite_version)



SQLite version: 3.45.1


In [ ]:
# DDL  = create, drop, derfines structure
# DML = inserct,select, update, delete works w e data
# primary key = used to identify each record
# foreign key = connects one table to another
# ref = specifices which table & column a foreign key links to
# the rest stores numbers / texts as data

run("""
DROP TABLE IF EXISTS drivers;
DROP TABLE IF EXISTS team;



CREATE TABLE team (
    id             INTEGER PRIMARY KEY,
    team_name      TEXT,
    team_principal TEXT
);

CREATE TABLE drivers (
    id      INTEGER PRIMARY KEY,
    name    TEXT,
    age     INTEGER,
    races   INTEGER,
    team_id INTEGER REFERENCES team(id)
);

INSERT INTO team VALUES
    (1, 'Red Bull', 'Laurent Mekies'),
    (2, 'Ferrari',  'Fred Vasseur'),
    (3, 'McLaren',  'Andrea Stella');

INSERT INTO drivers VALUES
    (1,  'Max Verstappen',   28, 227, 1),
    (3,  'Daniel Ricciardo', 36, 257, 1),
    (16, 'Charles Leclerc',  27, 165, 2),
    (44, 'Lewis Hamilton',   40, 374, 2),
    (81, 'Oscar Piastri',    24,  64, 3);
""")
print("Tables created")


Tables created


In [ ]:
# practicing with SELECT & FROM

sql("SELECT * from drivers")

,id,name,age,races,team_id
0,1,Max Verstappen,28,227,1
1,3,Daniel Ricciardo,36,257,1
2,16,Charles Leclerc,27,165,2
3,44,Lewis Hamilton,40,374,2
4,81,Oscar Piastri,24,64,3


In [ ]:
sql("SELECT * FROM team")

,id,team_name,team_principal
0,1,Red Bull,Laurent Mekies
1,2,Ferrari,Fred Vasseur
2,3,McLaren,Andrea Stella


In [ ]:
sql("SELECT name AS driver, races AS race_starts FROM drivers")

,driver,race_starts
0,Max Verstappen,227
1,Daniel Ricciardo,257
2,Charles Leclerc,165
3,Lewis Hamilton,374
4,Oscar Piastri,64


In [ ]:
# filtering with WHERE
# WHERE keeps only the rows that satisfy a condition. You can combine conditions with AND, OR and NOT, use comparison operators (= != > < >= <=)
# test membership with IN or ranges with BETWEEN

sql("SELECT name, races FROM drivers WHERE races>150")



,name,races
0,Max Verstappen,227
1,Daniel Ricciardo,257
2,Charles Leclerc,165
3,Lewis Hamilton,374


In [ ]:
sql("SELECT name, age, races FROM drivers WHERE races > 100 AND age < 30")

,name,age,races
0,Max Verstappen,28,227
1,Charles Leclerc,27,165


In [ ]:
# IN filters & selects from a list of values
# BETWEEN is inclusive of both ends

sql("""
SELECT name, team_id
FROM drivers
WHERE team_id IN (1, 3)
AND age BETWEEN 20 and 30
""")

,name,team_id
0,Max Verstappen,1
1,Oscar Piastri,3


In [ ]:
# '_% = filters what starts w m, % = any number of characters
# '%son' = filters what ends w '__'
# '%_%' = filters whatever contains _
# '_#%' = filters out any single character, then "ax"

In [ ]:
sql("SELECT * FROM drivers WHERE name LIKE 'M%'")

,id,name,age,races,team_id
0,1,Max Verstappen,28,227,1


In [ ]:
sql("SELECT name FROM drivers WHERE name LIKE '%ar%'")

,name
0,Daniel Ricciardo
1,Charles Leclerc
2,Oscar Piastri


In [ ]:
# sorting & limiting by ORDER BY & LIMIT _

# Top 3 most experienced drivers
sql("""
SELECT name, races
FROM drivers
ORDER BY races DESC
LIMIT 3
""")

In [ ]:
sql("SELECT name, team_id, age FROM drivers ORDER BY team_id, age DESC")

,name,team_id,age
0,Daniel Ricciardo,1,36
1,Max Verstappen,1,28
2,Lewis Hamilton,2,40
3,Charles Leclerc,2,27
4,Oscar Piastri,3,24


In [ ]:
# summarising data w aggregate functions

# col means column
# COUNT(*) = number of rows
# SUM(col) = total of a numeric column
# AVG(col) = average
# MIN(col)/MAX(col) = smallest/largest value
# ROUND(AVG(age), 1) = rounds to __ d.p.

In [ ]:


sql("""
SELECT COUNT(*)            AS drivers,
       SUM(races)          AS total_races,
       ROUND(AVG(age), 1)  AS avg_age,
       MIN(age)            AS youngest,
       MAX(races)          AS most_races
FROM drivers
""")

,drivers,total_races,avg_age,youngest,most_races
0,5,1087,31.0,24,374


In [ ]:
# Grouping with GROUP BY and HAVING
# HAVING filters groups after they are aggregated
# whereas WHERE filters rows before grouping
# GROUP BY seperates drivers into teams = multiple rows as well, without it = calculates for entire table

# Average race starts and age per team
sql("""
SELECT team_id,
       COUNT(*)              AS drivers,
       ROUND(AVG(races), 1)  AS avg_races,
       ROUND(AVG(age), 1)    AS avg_age

FROM drivers
GROUP BY team_id
""")

,team_id,drivers,avg_races,avg_age
0,1,2,242.0,32.0
1,2,2,269.5,33.5
2,3,1,64.0,24.0


In [ ]:
# Only teams whose drivers average more than 200 races
sql("""
SELECT team_id, AVG(races) AS avg_races
FROM drivers
GROUP BY team_id
HAVING AVG(races) > 200
""")

,team_id,avg_races
0,1,242.0
1,2,269.5


In [ ]:
sql("""
SELECT d.name
FROM drivers d JOIN team t ON d.team_id = t.id
WHERE t.team_name = 'Red Bull'
""")


,name
0,Max Verstappen
1,Daniel Ricciardo


In [ ]:
sql("SELECT name, age FROM drivers WHERE age = (SELECT MIN(age) FROM drivers)")

,name,age
0,Oscar Piastri,24


In [ ]:
sql("""
SELECT t.team_name, ROUND(AVG(d.age), 1) AS avg_age
FROM drivers d JOIN team t ON d.team_id = t.id
GROUP BY t.team_name
ORDER BY avg_age DESC
""")

,team_name,avg_age
0,Ferrari,33.5
1,Red Bull,32.0
2,McLaren,24.0
